# Laboratorio: eliminación de Gauss–Jordan

En este laboratorio trabajaremos con aritmética exacta mediante `sympy`. El objetivo no es sustituir el procedimiento manual, sino verificarlo, explorar sistemas con parámetros y comprobar las conclusiones mediante sustitución.

Al terminar podrá:

1. calcular la forma escalonada reducida y localizar pivotes;
2. distinguir sistemas con solución única, infinitas soluciones o sin solución;
3. reproducir operaciones por filas con matrices elementales;
4. recuperar relaciones entre las columnas originales;
5. clasificar un sistema que depende de parámetros.

In [ ]:
import sympy as sp

sp.init_printing()

## 1. Aritmética exacta y una función de diagnóstico

Con números enteros y racionales, `sympy` conserva fracciones exactas. Esto evita que un cero matemático aparezca como un decimal muy pequeño por redondeo.

La función siguiente calcula la forma reducida de $[A\mid b]$, los rangos y el tipo de solución. Observe que el número de incógnitas es el número de columnas de $A$, no el número de filas.

In [ ]:
def diagnostico_sistema(A, b):
    A = sp.Matrix(A)
    b = sp.Matrix(b)
    aumentada = A.row_join(b)
    reducida, pivotes_aumentada = aumentada.rref()
    rango_A = A.rank()
    rango_aumentada = aumentada.rank()
    n = A.cols

    if rango_A < rango_aumentada:
        tipo = "sin solución"
    elif rango_A == n:
        tipo = "solución única"
    else:
        tipo = f"infinitas soluciones ({n - rango_A} variable(s) libre(s))"

    return {
        "A": A,
        "b": b,
        "aumentada": aumentada,
        "reducida": reducida,
        "pivotes_aumentada": pivotes_aumentada,
        "rango_A": rango_A,
        "rango_aumentada": rango_aumentada,
        "tipo": tipo,
    }


def mostrar_diagnostico(datos):
    display(datos["aumentada"])
    display(datos["reducida"])
    print("rango(A) =", datos["rango_A"])
    print("rango([A|b]) =", datos["rango_aumentada"])
    print("clasificación:", datos["tipo"])

## 2. Reducción paso a paso

La forma reducida por sí sola no muestra el procedimiento. La siguiente función conserva, después de cada operación, una copia exacta de la matriz y una descripción de la operación realizada.

El algoritmo elige el primer candidato no nulo disponible en cada columna. En problemas numéricos con decimales sería necesario incorporar una tolerancia y una estrategia de pivoteo; aquí trabajamos con expresiones exactas.

In [ ]:
def gauss_jordan_paso_a_paso(A):
    M = sp.Matrix(A)
    filas, columnas = M.shape
    pasos = []
    fila_pivote = 0

    for columna in range(columnas):
        if fila_pivote >= filas:
            break

        candidato = next(
            (i for i in range(fila_pivote, filas) if M[i, columna] != 0),
            None,
        )
        if candidato is None:
            continue

        if candidato != fila_pivote:
            M.row_swap(candidato, fila_pivote)
            pasos.append(
                (f"F{fila_pivote + 1} <-> F{candidato + 1}", M.copy())
            )

        pivote = sp.simplify(M[fila_pivote, columna])
        if pivote != 1:
            M.row_op(
                fila_pivote,
                lambda valor, j: sp.simplify(valor / pivote),
            )
            pasos.append(
                (f"F{fila_pivote + 1} <- (1/({pivote})) F{fila_pivote + 1}",
                 M.copy())
            )

        for i in range(filas):
            if i == fila_pivote or M[i, columna] == 0:
                continue
            coeficiente = sp.simplify(M[i, columna])
            M.row_op(
                i,
                lambda valor, j: sp.simplify(
                    valor - coeficiente * M[fila_pivote, j]
                ),
            )
            pasos.append(
                (f"F{i + 1} <- F{i + 1} - ({coeficiente}) F{fila_pivote + 1}",
                 M.copy())
            )

        fila_pivote += 1

    return pasos, M

In [ ]:
M = sp.Matrix([[1, 1, 3], [2, -1, 0]])
pasos, M_reducida = gauss_jordan_paso_a_paso(M)

display(M)
for numero, (operacion, matriz) in enumerate(pasos, start=1):
    print(f"Paso {numero}: {operacion}")
    display(matriz)

assert M_reducida == M.rref()[0]

## 2. Los tres tipos de sistema

### 2.1 Solución única

Resolvemos

$$
\begin{aligned}
x+y&=3,\\
2x-y&=0.
\end{aligned}
$$

In [ ]:
A_unica = sp.Matrix([[1, 1], [2, -1]])
b_unica = sp.Matrix([3, 0])
d_unica = diagnostico_sistema(A_unica, b_unica)
mostrar_diagnostico(d_unica)

x_unica = A_unica.LUsolve(b_unica)
print("solución:")
display(x_unica)
print("residuo Ax-b:")
display(A_unica * x_unica - b_unica)

### 2.2 Infinitas soluciones

En el siguiente sistema hay tres incógnitas y solamente dos pivotes. `linsolve` conserva el parámetro libre y entrega el conjunto completo de soluciones.

In [ ]:
A_inf = sp.Matrix([
    [1, 1, 1],
    [2, 2, 2],
    [1, -1, 1],
])
b_inf = sp.Matrix([2, 4, 0])
d_inf = diagnostico_sistema(A_inf, b_inf)
mostrar_diagnostico(d_inf)

x1, x2, x3 = sp.symbols("x1 x2 x3")
sol_inf = sp.linsolve((A_inf, b_inf), (x1, x2, x3))
print("conjunto solución:")
display(sol_inf)

### 3.3 Interpretación geométrica de infinitas soluciones

El sistema homogéneo

$$
\begin{aligned}
x+y+z&=0,\\
x+y+2z&=0
\end{aligned}
$$

representa dos planos que pasan por el origen. Al restar las ecuaciones se obtiene $z=0$ y luego $y=-x$. Su intersección es la recta

$$
(x,y,z)=t(1,-1,0),\qquad t\in\mathbb R.
$$

La figura permite ver que “infinitas soluciones” no significa todo $\mathbb R^3$: el conjunto solución puede ser una recta o un plano.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rejilla = np.linspace(-3, 3, 35)
X, Y = np.meshgrid(rejilla, rejilla)
Z1 = -X - Y
Z2 = (-X - Y) / 2

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(X, Y, Z1, alpha=0.35, color="tab:blue")
ax.plot_surface(X, Y, Z2, alpha=0.35, color="tab:orange")

t = np.linspace(-3, 3, 100)
ax.plot(t, -t, np.zeros_like(t), color="crimson", linewidth=3,
        label=r"$t(1,-1,0)$")
ax.set(xlabel="x", ylabel="y", zlabel="z",
       title="Intersección de dos planos")
ax.legend()
plt.show()

direccion = sp.Matrix([1, -1, 0])
A_planos = sp.Matrix([[1, 1, 1], [1, 1, 2]])
assert A_planos * direccion == sp.zeros(2, 1)
assert A_planos.rank() == 2

### 2.3 Sistema incompatible

La última columna de la matriz aumentada se convierte en columna pivote. La fila correspondiente representa una contradicción.

In [ ]:
A_sin = sp.Matrix([[1, 1], [2, 2]])
b_sin = sp.Matrix([1, 3])
d_sin = diagnostico_sistema(A_sin, b_sin)
mostrar_diagnostico(d_sin)

assert sp.linsolve((A_sin, b_sin)) is sp.EmptySet

## 3. Operaciones elementales como matrices

Partimos de

$$
A=\begin{bmatrix}1&1\\2&-1\end{bmatrix}.
$$

Aplicaremos las mismas operaciones del ejemplo manual:

1. $F_2\leftarrow F_2-2F_1$;
2. $F_2\leftarrow-\frac13F_2$;
3. $F_1\leftarrow F_1-F_2$.

Cada matriz elemental se obtiene aplicando la operación correspondiente a $I_2$.

In [ ]:
A = sp.Matrix([[1, 1], [2, -1]])
E1 = sp.Matrix([[1, 0], [-2, 1]])
E2 = sp.diag(1, sp.Rational(-1, 3))
E3 = sp.Matrix([[1, -1], [0, 1]])

A1 = E1 * A
A2 = E2 * A1
A3 = E3 * A2

display(A, A1, A2, A3)
assert A3 == sp.eye(2)

E = E3 * E2 * E1
print("Producto de matrices elementales:")
display(E)
print("Inversa calculada por Gauss–Jordan:")
display(A.inv())
assert E == A.inv()
assert A * E == sp.eye(2) and E * A == sp.eye(2)

## 4. Pivotes y relaciones entre columnas

Reducimos la matriz usada en las notas de clase. Los índices que devuelve `sympy` comienzan en cero; por eso los convertimos a numeración matemática sumando uno.

In [ ]:
B = sp.Matrix([
    [1, 1, 4, 4, 18],
    [1, 1, 4, 1, 6],
    [-1, 1, 0, 2, 12],
    [1, 1, 4, 1, 6],
])

B_rref, pivotes = B.rref()
display(B)
display(B_rref)
print("columnas pivote (numeración matemática):", tuple(j + 1 for j in pivotes))
assert pivotes == (0, 1, 3)

In [ ]:
v1, v2, v3, v4, v5 = [B[:, j] for j in range(B.cols)]

relacion_v3 = sp.simplify(v3 - (2 * v1 + 2 * v2))
relacion_v5 = sp.simplify(v5 - (-v1 + 3 * v2 + 4 * v4))

print("v3 - (2v1+2v2) =")
display(relacion_v3)
print("v5 - (-v1+3v2+4v4) =")
display(relacion_v5)

assert relacion_v3 == sp.zeros(B.rows, 1)
assert relacion_v5 == sp.zeros(B.rows, 1)

## 5. Sistema con parámetros

Considere

$$
\begin{aligned}
x+z&=q,\\
y+2w&=0,\\
x+2z+3w&=0,\\
2y+3z+pw&=3.
\end{aligned}
$$

De las tres primeras ecuaciones:

$$
x=q-z,\qquad y=-2w,\qquad z=-q-3w.
$$

Al sustituir en la cuarta se obtiene la ecuación decisiva

$$
(p-13)w=3(q+1).
$$

Por tanto:

- si $p\neq13$, hay solución única;
- si $p=13$ y $q=-1$, hay infinitas soluciones;
- si $p=13$ y $q\neq-1$, no hay solución.

Comprobamos un representante de cada caso mediante rangos.

In [ ]:
def sistema_parametrico(p, q):
    A = sp.Matrix([
        [1, 0, 1, 0],
        [0, 1, 0, 2],
        [1, 0, 2, 3],
        [0, 2, 3, p],
    ])
    b = sp.Matrix([q, 0, 0, 3])
    return diagnostico_sistema(A, b)


casos = {
    "p distinto de 13": sistema_parametrico(5, 2),
    "p=13, q=-1": sistema_parametrico(13, -1),
    "p=13, q distinto de -1": sistema_parametrico(13, 0),
}

for nombre, datos in casos.items():
    print(nombre, "->", datos["tipo"],
          f"(rangos {datos['rango_A']}, {datos['rango_aumentada']})")

assert casos["p distinto de 13"]["tipo"] == "solución única"
assert casos["p=13, q=-1"]["tipo"].startswith("infinitas soluciones")
assert casos["p=13, q distinto de -1"]["tipo"] == "sin solución"

## 6. Ejercicios de laboratorio

1. Modifique el sistema con infinitas soluciones para que tenga dos variables libres. Describa el conjunto solución en forma paramétrica vectorial.
2. Construya una matriz elemental que intercambie $F_1$ y $F_3$ en una matriz de tres filas. Verifique el efecto mediante multiplicación.
3. Reduzca

   $$
   \begin{bmatrix}
   4&6&5&7&3\\
   4&7&4&6&2\\
   4&6&4&6&4\\
   2&3&2&3&2
   \end{bmatrix}
   $$

   y compare los pivotes con los obtenidos manualmente.
4. En la matriz $B$, use `gauss_jordan_solve` o `linsolve` para recuperar automáticamente los coeficientes que expresan $v_3$ y $v_5$ mediante $v_1,v_2,v_4$.
5. Para $p\neq13$, encuentre la solución del sistema paramétrico como función de $p$ y $q$, y verifíquela sustituyendo en las cuatro ecuaciones.

```{admonition} Criterio de entrega
:class: tip
Cada respuesta computacional debe incluir una interpretación: pivotes, variables libres, rango, clasificación o verificación por residuo, según corresponda.
```